<a href="https://colab.research.google.com/github/AlexAmaya19/C-digoSeguridadPI/blob/main/SadTalker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title **setup（about 5 minutes）**
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.8 2
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.9 1
!python --version
!apt-get update
!apt install software-properties-common
!sudo dpkg --remove --force-remove-reinstreq python3-pip python3-setuptools python3-wheel
!apt-get install python3-pip

print('Git clone project and install requirements...')
!git clone https://github.com/cedro3/SadTalker.git &> /dev/null
%cd SadTalker
!export PYTHONPATH=/content/SadTalker:$PYTHONPATH
!python3.8 -m pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113
!apt update
!apt install ffmpeg &> /dev/null
!python3.8 -m pip install -r requirements.txt


update-alternatives: error: alternative path /usr/bin/python3.8 doesn't exist
update-alternatives: error: alternative path /usr/bin/python3.9 doesn't exist
Python 3.13.15
Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [75.3 kB]
Get:7 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,934 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:10 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu no

In [2]:
#@title **download model（about 1 minute)**
print('Download pre-trained models...')
!rm -rf checkpoints
!bash scripts/download_models.sh

Download pre-trained models...
--2026-10-19 18:05:31--  https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar
Resolving github.com (github.com)... 20.29.134.23
Connecting to github.com (github.com)|20.29.134.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/569518584/ccc415aa-c6f4-47ee-8250-b10bf440ba62?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-19T18%3A49%3A01Z&rscd=attachment%3B+filename%3Dmapping_00109-model.pth.tar&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-10-19T17%3A48%3A19Z&ske=2026-10-19T18%3A49%3A01Z&sks=b&skv=2018-11-09&sig=X44Lk3dPFcbMrqK4aREHfNz7sbzraxTncNFeK91vfjA%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5MjQzNjczMSwibmJmIjo

In [24]:
#@title **inference for face**
image ='AngloSaxons.jpg' #@param {type:"string"}
audio ='AngloSaxon.m4a' #@param {type:"string"}
source_image = 'examples/source_image/' + image
driven_audio = 'examples/driven_audio/' + audio

!python3 inference.py --driven_audio $driven_audio \
           --source_image $source_image \
           --result_dir ./results \
           --preprocess full \
           --still

using safetensor as default
3DMM Extraction for source image
landmark Det:: 100% 1/1 [00:00<00:00,  7.39it/s]
3DMM Extraction In Video:: 100% 1/1 [00:00<00:00, 18.15it/s]
mel:: 100% 4150/4150 [00:00<00:00, 51513.35it/s]
audio2exp:: 100% 415/415 [00:00<00:00, 450.80it/s]
Face Renderer:: 100% 2075/2075 [39:01<00:00,  1.13s/it]
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (256, 253) to (256, 256) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
The generated video is named ./results/2026_10_19_18.26.20/AngloSaxons##AngloSaxon.mp4
OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
seamlessClone:: 100% 4150/4150 [00:48<00:00, 84.90it/s]
The generated video is named ./results/2026_

In [22]:
p = '/content/SadTalker/src/face3d/util/preprocess.py'
s = open(p).read()
s = s.replace("trans_params = np.array([w0, h0, s, t[0], t[1]])",
              "trans_params = np.array([float(np.squeeze(w0)), float(np.squeeze(h0)), float(np.squeeze(s)), float(np.squeeze(t[0])), float(np.squeeze(t[1]))])")
open(p, 'w').write(s)

3442

In [20]:
!sed -i -E 's/np\.float\b/float/g; s/np\.int\b/int/g; s/np\.bool\b/bool/g' /usr/local/lib/python3.13/dist-packages/facexlib/alignment/awing_arch.py
!grep -rlE "np\.(float|int|bool)\b" /usr/local/lib/python3.13/dist-packages/facexlib /content/SadTalker | xargs -r sed -i -E 's/np\.float\b/float/g; s/np\.int\b/int/g; s/np\.bool\b/bool/g'

In [18]:
!ls -la /content/SadTalker/examples/source_image/
!ls -la /content/SadTalker/examples/driven_audio/
!ls -la /content

total 23028
drwxr-xr-x 3 root root    4096 Oct 19 18:11 .
drwxr-xr-x 5 root root    4096 Oct 19 17:57 ..
-rw-r--r-- 1 root root  139326 Oct 19 18:11 AngloSaxons.jpg
-rw-r--r-- 1 root root  750139 Oct 19 17:57 art_0.png
-rw-r--r-- 1 root root  569012 Oct 19 17:57 art_10.png
-rw-r--r-- 1 root root  488957 Oct 19 17:57 art_11.png
-rw-r--r-- 1 root root  721100 Oct 19 17:57 art_12.png
-rw-r--r-- 1 root root  632028 Oct 19 17:57 art_13.png
-rw-r--r-- 1 root root  650288 Oct 19 17:57 art_14.png
-rw-r--r-- 1 root root  673057 Oct 19 17:57 art_15.png
-rw-r--r-- 1 root root 1480263 Oct 19 17:57 art_16.png
-rw-r--r-- 1 root root 2092096 Oct 19 17:57 art_17.png
-rw-r--r-- 1 root root  117636 Oct 19 17:57 art_18.png
-rw-r--r-- 1 root root  472941 Oct 19 17:57 art_19.png
-rw-r--r-- 1 root root  489361 Oct 19 17:57 art_1.png
-rw-r--r-- 1 root root  710209 Oct 19 17:57 art_20.png
-rw-r--r-- 1 root root  831523 Oct 19 17:57 art_2.png
-rw-r--r-- 1 root root 1353396 Oct 19 17:57 art_3.png
-rw-r--r-- 1 r

In [16]:
p = '/content/SadTalker/src/utils/face_enhancer.py'
s = open(p).read()
s = s.replace("from gfpgan import GFPGANer", "try:\n    from gfpgan import GFPGANer\nexcept Exception:\n    GFPGANer = None")
open(p, 'w').write(s)

4483

In [12]:
!pip install kornia
!pip install yacs
!pip install face_alignment
!pip install resampy
!pip install pydub
!pip install basicsr-fixed

  Using cached kornia-0.8.3-py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 114.5 MB/s eta 0:00:00
  Using cached yacs-0.1.8-py3-none-any.whl.metadata (639 bytes)
  Using cached face_alignment-1.5.0-py3-none-any.whl.metadata (8.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.1 MB/s eta 0:00:00
  Using cached resampy-0.4.3-py3-none-any.whl.metadata (3.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 46.7 MB/s eta 0:00:00
  Using cached basicsr-fixed-1.4.2.tar.gz (173 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above f

In [7]:
!grep -rl "np.VisibleDeprecationWarning" /content/SadTalker | xargs sed -i 's/np\.VisibleDeprecationWarning/np.exceptions.VisibleDeprecationWarning/g'

In [10]:
!pip install kornia yacs pydub safetensors face_alignment resampy librosa imageio imageio-ffmpeg basicsr-fixed
!pip install facexlib gfpgan --no-deps
!pip install filterpy

  Using cached kornia-0.8.3-py3-none-any.whl.metadata (22 kB)
  Using cached yacs-0.1.8-py3-none-any.whl.metadata (639 bytes)
  Using cached face_alignment-1.5.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached resampy-0.4.3-py3-none-any.whl.metadata (3.0 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 8.1 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
  Using cached facexlib-0.3.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached gfpgan-1.3.8-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59

In [ ]:
#@title **play movie**
import glob
from IPython.display import HTML
from base64 import b64encode
import os, sys

# get the last from results
mp4_name = sorted(glob.glob('./results/*.mp4'))[-1]

mp4 = open('{}'.format(mp4_name),'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

print('Display animation: {}'.format(mp4_name), file=sys.stderr)
display(HTML("""
  <video width=256 controls>
        <source src="%s" type="video/mp4">
  </video>
  """ % data_url))

In [ ]:
#@title **inference for portrait**
image ='full3.png' #@param {type:"string"}
audio ='eluosi.wav' #@param {type:"string"}
source_image = 'examples/source_image/' + image
driven_audio = 'examples/driven_audio/' + audio

!python3.8 inference.py --driven_audio $driven_audio \
           --source_image $source_image \
           --result_dir ./results --still --preprocess full --enhancer gfpgan

In [ ]:
#@title **play movie**
import glob
from IPython.display import HTML
from base64 import b64encode
import os, sys

# get the last from results
mp4_name = sorted(glob.glob('./results/*.mp4'))[-1]

mp4 = open('{}'.format(mp4_name),'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

print('Display animation: {}'.format(mp4_name), file=sys.stderr)
display(HTML("""
  <video width=256 controls>
        <source src="%s" type="video/mp4">
  </video>
  """ % data_url))